# Phase 8: Domain Adaptation Results Analysis (DA only)

Goals:
- DA performance analysis
- validate predicted vs actual transferability

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Path settings
ROOT = r'..'
RESULTS_DIR = os.path.join(ROOT, 'results')
BENCHMARK_DIR = os.path.join(ROOT, 'benchmark')
BENCH_RESULTS_DIR = os.path.join(BENCHMARK_DIR, 'results')
os.makedirs(BENCH_RESULTS_DIR, exist_ok=True)

print("="*100)
print("PHASE 8: Domain Adaptation Results Analysis")
print("="*100)
print(f"Results: {BENCH_RESULTS_DIR}\n")

PHASE 8: Domain Adaptation Results Analysis
Results: ..\benchmark\results



In [2]:
# Load DA results
print("Loading DA results...\n")

da_dir = os.path.join(RESULTS_DIR, 'mmd_adaptation')
da_results = {}

properties = ['clearance', 'half_life', 'fu']

for prop in properties:
    da_path = os.path.join(da_dir, f'{prop}.csv')
    if os.path.exists(da_path):
        df = pd.read_csv(da_path)
        da_results[prop] = df
        print(f"✓ Loaded DA: {prop}")
    else:
        print(f"✗ DA not found: {prop}")

print()

Loading DA results...

✓ Loaded DA: clearance
✓ Loaded DA: half_life
✓ Loaded DA: fu



In [3]:
# Load transferability profile
print("Loading transferability profiles...\n")

try:
    with open(os.path.join(BENCH_RESULTS_DIR, 'phase_07_transferability_profile.json'), 'r') as f:
        transferability_profiles = json.load(f)
    print(f"✓ Loaded transferability profiles\n")
except:
    print(f"✗ Transferability profiles not found\n")
    transferability_profiles = {}

Loading transferability profiles...

✓ Loaded transferability profiles



In [4]:
# DA Results Analysis
da_analysis = {}

for prop in properties:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    if prop not in da_results:
        print(f"  No DA results")
        continue
    
    da_df = da_results[prop]
    
    print(f"\n  DA Results (MMD-based):")
    print(f"    n_folds: {len(da_df)}")
    
    da_metrics = {}
    for metric in ['r', 'rho', 'r2', 'rmse', 'mae', 'mbe']:
        if metric in da_df.columns:
            mean = da_df[metric].mean()
            std = da_df[metric].std()
            da_metrics[metric] = {'mean': mean, 'std': std}
            print(f"    {metric}: {mean:.4f} ± {std:.4f}")
    
    # Compare with prediction
    if prop in transferability_profiles:
        profile = transferability_profiles[prop]
        predicted_success_prob = profile.get('da_success_probability', None)
        predicted_score = profile.get('overall_score', None)
        
        # Actual DA performance
        actual_r2 = da_metrics.get('r2', {}).get('mean', None)
        actual_rho = da_metrics.get('rho', {}).get('mean', None)
        
        print(f"\n  Prediction vs Actual:")
        print(f"    Predicted success prob: {predicted_success_prob:.0%}" if predicted_success_prob else "    Predicted success prob: N/A")
        print(f"    Predicted score: {predicted_score}/100" if predicted_score else "    Predicted score: N/A")
        print(f"    Actual R²: {actual_r2:.4f}" if actual_r2 else "    Actual R²: N/A")
        print(f"    Actual Spearman ρ: {actual_rho:.4f}" if actual_rho else "    Actual Spearman ρ: N/A")
        
        da_analysis[prop] = {
            'da_metrics': {k: {mk: float(mv) if isinstance(mv, (int, float, np.number)) else mv for mk, mv in v.items()} for k, v in da_metrics.items()},
            'predicted_success_prob': float(predicted_success_prob) if predicted_success_prob else None,
            'predicted_score': int(predicted_score) if predicted_score else None,
            'actual_r2': float(actual_r2) if actual_r2 else None,
            'actual_rho': float(actual_rho) if actual_rho else None,
        }
    else:
        da_analysis[prop] = {'da_metrics': {k: {mk: float(mv) if isinstance(mv, (int, float, np.number)) else mv for mk, mv in v.items()} for k, v in da_metrics.items()}}


Property: CLEARANCE

  DA Results (MMD-based):
    n_folds: 5
    r: 0.3592 ± 0.0703
    rho: 0.3736 ± 0.0772
    r2: 0.0538 ± 0.0790
    rmse: 0.9666 ± 0.0730
    mae: 0.7195 ± 0.0327
    mbe: -0.0646 ± 0.0518

  Prediction vs Actual:
    Predicted success prob: 20%
    Predicted score: 44/100
    Actual R²: 0.0538
    Actual Spearman ρ: 0.3736

Property: HALF_LIFE

  DA Results (MMD-based):
    n_folds: 5
    r: 0.3713 ± 0.0654
    rho: 0.3911 ± 0.0329
    r2: -0.0015 ± 0.1770
    rmse: 0.9885 ± 0.0809
    mae: 0.7427 ± 0.0492
    mbe: 0.0507 ± 0.1423

  Prediction vs Actual:
    Predicted success prob: 70%
    Predicted score: 71/100
    Actual R²: -0.0015
    Actual Spearman ρ: 0.3911

Property: FU

  DA Results (MMD-based):
    n_folds: 5
    r: 0.5756 ± 0.0738
    rho: 0.5989 ± 0.0812
    r2: 0.1473 ± 0.2622
    rmse: 0.8887 ± 0.0900
    mae: 0.6951 ± 0.0698
    mbe: 0.0303 ± 0.2826

  Prediction vs Actual:
    Predicted success prob: 20%
    Predicted score: 48/100
    Actual R

In [5]:
# Summary Report
print(f"\n\n{'='*100}")
print("DA PERFORMANCE SUMMARY")
print(f"{'='*100}\n")

summary_data = []

for prop, result in da_analysis.items():
    metrics = result.get('da_metrics', {})
    row = {
        'Property': prop,
        'Spearman_ρ': f"{metrics.get('rho', {}).get('mean', 0):.4f}",
        'R2': f"{metrics.get('r2', {}).get('mean', 0):.4f}",
        'RMSE': f"{metrics.get('rmse', {}).get('mean', 0):.4f}",
        'Predicted_Prob': f"{result.get('predicted_success_prob', 0):.0%}",
    }
    summary_data.append(row)

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
else:
    print("No DA results available")
print()



DA PERFORMANCE SUMMARY

 Property Spearman_ρ      R2   RMSE Predicted_Prob
clearance     0.3736  0.0538 0.9666            20%
half_life     0.3911 -0.0015 0.9885            70%
       fu     0.5989  0.1473 0.8887            20%



In [6]:
# Fold-by-fold analysis
print(f"\n{'='*100}")
print("FOLD-BY-FOLD ANALYSIS")
print(f"{'='*100}\n")

for prop in properties:
    if prop not in da_results:
        continue
    
    da_df = da_results[prop].copy()
    
    print(f"\n{prop.upper()}:")
    print(f"  {'Fold':<6} {'R²':<12} {'Spearman ρ':<12} {'RMSE':<12}")
    print(f"  {'-'*42}")
    
    for fold_idx in range(len(da_df)):
        r2 = da_df.iloc[fold_idx]['r2'] if 'r2' in da_df.columns else 0
        rho = da_df.iloc[fold_idx]['rho'] if 'rho' in da_df.columns else 0
        rmse = da_df.iloc[fold_idx]['rmse'] if 'rmse' in da_df.columns else 0
        
        print(f"  {fold_idx:<6} {r2:<12.4f} {rho:<12.4f} {rmse:<12.4f}")


FOLD-BY-FOLD ANALYSIS


CLEARANCE:
  Fold   R²           Spearman ρ   RMSE        
  ------------------------------------------
  0      -0.0111      0.3430       0.9889      
  1      -0.0500      0.2606       1.0109      
  2      0.0934       0.3944       0.8883      
  3      0.1120       0.4020       0.8928      
  4      0.1246       0.4679       1.0520      

HALF_LIFE:
  Fold   R²           Spearman ρ   RMSE        
  ------------------------------------------
  0      0.1266       0.3927       0.9664      
  1      -0.0883      0.3857       1.0621      
  2      0.1949       0.4306       0.8666      
  3      0.0112       0.4057       1.0620      
  4      -0.2520      0.3410       0.9853      

FU:
  Fold   R²           Spearman ρ   RMSE        
  ------------------------------------------
  0      0.1865       0.5379       0.9694      
  1      -0.2640      0.5814       0.9137      
  2      0.4619       0.7393       0.7470      
  3      0.1332       0.5492       0.9543   

In [7]:
# Save results
with open(os.path.join(BENCH_RESULTS_DIR, 'phase_08_da_benchmark.json'), 'w') as f:
    json.dump(da_analysis, f, indent=2)

print(f"\n\n✓ Results saved to benchmark/results/")
print(f"  - phase_08_da_benchmark.json")
print("\nPhase 8 Complete!")
print("\nNote: Baseline (target-only) results not found.")
print("      To run baseline: execute 01_model/01_baseline_target_only.ipynb")



✓ Results saved to benchmark/results/
  - phase_08_da_benchmark.json

Phase 8 Complete!

Note: Baseline (target-only) results not found.
      To run baseline: execute 01_model/01_baseline_target_only.ipynb
